# Ablación con varias semillas (R3, R4, R6) — Detección de violencia en video
**MIA-203 Redes Neuronales y Aprendizaje Profundo — UNI FIIS**  
Equipo: Alvaro Alejandro Siesquén Abad · Kiara Isumi Alvarez Barrionuevo · Sergio Alonso Ortiz Pauca · Alfonso Pérez Ramos

Este cuaderno reproduce la **Tabla I del artículo**: 6 variantes del modelo × 3 semillas, con la partición fija (semilla 42) del cuaderno principal `Proyecto_violencia_Lima.ipynb`.

| Variante | Módulo temporal |
|---|---|
| `baseline_avg` | Promedio temporal (línea base, sin memoria) |
| `resnet_rnn` | RNN simple |
| `resnet_lstm` | LSTM unidireccional |
| `resnet_bilstm` | BiLSTM sin atención |
| `resnet_bilstm_attn` | BiLSTM + atención temporal (propuesto) |
| `resnet_bilstm_attn` + `robusto` | Propuesto + aumento de degradación CCTV |

**Tiempo en Colab T4:** ~13 min por corrida. Elige el modo en la celda 4:
- `"rapido"`: semilla 42, las 6 variantes (~80 min).
- `"completo"`: 3 semillas, 18 corridas (~4 h).
- `"tabla"`: solo regenera las tablas con los resultados ya incluidos en `results/runs/` (segundos).

*Entorno de ejecución → Cambiar tipo de entorno → GPU T4.*

## 1. Repositorio y dependencias

In [ ]:
!git clone -q https://github.com/AlfonsoPe/violencia-lima-dl.git
%cd violencia-lima-dl
!pip install -q kagglehub opencv-python-headless scikit-learn pandas pyyaml tqdm

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

## 2. Elegir modo de ejecución

In [ ]:
MODO = "rapido"      # "rapido" | "completo" | "tabla"
SEMILLAS = {"rapido": [42], "completo": [42, 43, 44], "tabla": []}[MODO]
VARIANTES = [("baseline_avg", ""), ("resnet_rnn", ""), ("resnet_lstm", ""), ("resnet_bilstm", ""),
             ("resnet_bilstm_attn", ""), ("resnet_bilstm_attn", "robusto")]
print(MODO, "->", len(SEMILLAS) * len(VARIANTES), "corridas")

## 3. Datos (RLVS + Hockey Fight + Movies Fight) y extracción de 16 fotogramas por video
Misma construcción y partición 70/15/15 a nivel de video que el cuaderno principal (semilla 42, 12 videos reservados para pruebas).

In [ ]:
import os, kagglehub
if MODO != "tabla":
    os.makedirs("datos", exist_ok=True)
    for slug in ["mohamedmustafa/real-life-violence-situations-dataset",
                 "yassershrief/hockey-fight-vidoes",
                 "naveenk903/movies-fight-detection-dataset"]:
        ruta = kagglehub.dataset_download(slug)
        destino = os.path.join("datos", slug.split("/")[1])
        if not os.path.exists(destino):
            os.symlink(ruta, destino)
        print(slug, "->", ruta)
    !python datos.py datos /content/cache

## 4. Entrenamiento de la ablación
Cada corrida guarda en `results/runs/<variante>_s<semilla>/`: `metricas.json` (test limpio, test CCTV sintético, métricas por fuente, minutos y VRAM), `historial.csv` y `predicciones_test.csv`.

In [ ]:
os.environ["WORKERS"] = "2"   # Colab tiene 2 CPUs
for s in SEMILLAS:
    for modo, robusto in VARIANTES:
        print(f"=== {modo} {robusto} semilla {s} ===")
        !python entrenar.py {modo} {s} {robusto} 2>&1 | tail -2

## 5. Tabla de ablación (media ± desv. sobre semillas) y horas de GPU (R6)

In [ ]:
!python tabla.py
from IPython.display import Markdown
display(Markdown(open("results/tabla_ablacion.md").read()))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
d = pd.read_csv("results/tabla_ablacion_runs.csv")
g = d.groupby("modelo", sort=False)[["f1", "f1_cctv"]].agg(["mean", "std"])
etiquetas = [m.replace("ResNet-50 + ", "").replace(" (propuesto)", "") for m in g.index]
x = range(len(g))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar([i - 0.2 for i in x], g[("f1", "mean")], 0.4, yerr=g[("f1", "std")], label="F1 test limpio")
ax.bar([i + 0.2 for i in x], g[("f1_cctv", "mean")], 0.4, yerr=g[("f1_cctv", "std")], label="F1 test CCTV sintético")
ax.set_xticks(list(x)); ax.set_xticklabels(etiquetas, rotation=20, ha="right"); ax.set_ylim(50, 100)
ax.set_ylabel("F1 (%)"); ax.legend(); ax.set_title("Ablación: media ± desv. sobre semillas")
plt.tight_layout(); plt.show()

**Lectura:** en el test limpio todas las variantes quedan alrededor del 98 % (diferencias del tamaño de la variación entre semillas). Bajo la degradación CCTV sintética, los modelos entrenados con video limpio caen a ~70–78 % de F1, mientras que el entrenamiento con aumento CCTV mantiene ~96 %.